# 03-2 Keyword Split

processed/ 하위 카테고리 폴더들을 파일명 첫 키워드 기준으로 서브폴더로 분리한다.  
`body/`는 변경하지 않는다.

**파일명 구조:** `{keyword}_{subtype}_{gender}_{action}_{variant}_{frame}.png`  
→ 첫 번째 `_` 앞 토큰을 키워드로 사용

**예시:**
```
arms/armour_plate_male_walk_0_0.png  →  arms/armour/armour_plate_male_walk_0_0.png
arms/bracers_male_walk_0_0.png       →  arms/bracers/bracers_male_walk_0_0.png
```

## Section 1 — 환경 설정

In [1]:
import shutil
from pathlib import Path
from collections import defaultdict

PROCESSED_DIR = Path('dataset/processed')
SKIP_CATEGORIES = {'body'}  # 변경하지 않을 카테고리

## Section 2 — 구조 분석 (Dry Run)

실제 이동 없이 각 카테고리의 키워드 분포를 확인한다.

In [2]:
def get_keyword_counts(category_dir: Path) -> dict[str, int]:
    counts = defaultdict(int)
    for f in category_dir.iterdir():
        if f.is_file() and f.suffix == '.png':
            keyword = f.name.split('_')[0]
            counts[keyword] += 1
    return dict(counts)

need_split   = []
already_ok   = []

for cat_dir in sorted(PROCESSED_DIR.iterdir()):
    if not cat_dir.is_dir():
        continue
    if cat_dir.name in SKIP_CATEGORIES:
        print(f"[SKIP] {cat_dir.name}/")
        continue

    # 카테고리 내 직접 파일만 대상 (이미 서브폴더로 이동된 파일 제외)
    counts = get_keyword_counts(cat_dir)

    if not counts:
        print(f"[OK  ] {cat_dir.name}/  (파일 없음 — 이미 분리됨)")
        already_ok.append(cat_dir.name)
        continue

    total = sum(counts.values())
    keywords = sorted(counts.keys())

    if len(keywords) == 1:
        print(f"[OK  ] {cat_dir.name}/  ({keywords[0]}: {total:,})")
        already_ok.append(cat_dir.name)
    else:
        kw_summary = ', '.join(f"{k}:{counts[k]:,}" for k in keywords)
        print(f"[SPLIT] {cat_dir.name}/  → {kw_summary}")
        need_split.append(cat_dir.name)

print(f"\n분리 필요: {len(need_split)}개  /  분리 불필요: {len(already_ok)}개")

[SPLIT] arms/  → armour:472, bracers:485, hands:1,014, wrists:985
[SPLIT] backpack/  → backpack:5,880, basket:2,314, jetpack:1,686, squarepack:5,920, straps:7,080
[SPLIT] beards/  → beard:1,644, mustache:2,072
[SKIP] body/
[SPLIT] cape/  → solid:327, tattered:327, trim:188
[SPLIT] dress/  → bodice:4,464, kimono:39,011, sash:4,472, slit:4,464
[SPLIT] eyes/  → cyclops:5,071, eyebrows:1,035, human:50,688
[SPLIT] facial/  → earrings:52,656, glasses:122,749, masks:18,720, monocle:31,150, patches:126,005
[SPLIT] feet/  → accessory:12,312, armour:2,960, boots:92,334, hoofs:7,912, sandals:23,034, shoes:92,900, slippers:17,450, socks:52,400
[SPLIT] hair/  → afro:352, balding:352, bangs:704, bangslong:352, bangslong2:516, bangsshort:352, bedhead:352, bob:704, braid:1,445, braid2:523, bunches:611, buzzcut:352, cornrows:352, cowlick:704, curls:2,249, curly:1,056, curtains:704, dreadlocks:704, extensions:7,580, flat:1,060, half:352, halfmessy:352, high:874, idol:352, jewfro:704, lob:352, long:4,110

## Section 3 — 폴더 재분류 실행

각 카테고리 내 파일을 첫 키워드 서브폴더로 이동한다.

In [3]:
def split_by_keyword(category_dir: Path) -> dict[str, int]:
    moved = defaultdict(int)

    files = [f for f in category_dir.iterdir() if f.is_file() and f.suffix == '.png']
    for f in files:
        keyword = f.name.split('_')[0]
        dest_dir = category_dir / keyword
        dest_dir.mkdir(exist_ok=True)
        shutil.move(str(f), str(dest_dir / f.name))
        moved[keyword] += 1

    return dict(moved)


total_moved = 0

for cat_name in need_split:
    cat_dir = PROCESSED_DIR / cat_name
    moved = split_by_keyword(cat_dir)
    total = sum(moved.values())
    total_moved += total
    kw_str = ', '.join(f"{k}:{v:,}" for k, v in sorted(moved.items()))
    print(f"{cat_name}/  ({total:,} files)  →  {kw_str}")

print(f"\n총 이동: {total_moved:,} files")

arms/  (2,956 files)  →  armour:472, bracers:485, hands:1,014, wrists:985
backpack/  (22,880 files)  →  backpack:5,880, basket:2,314, jetpack:1,686, squarepack:5,920, straps:7,080
beards/  (3,716 files)  →  beard:1,644, mustache:2,072
cape/  (842 files)  →  solid:327, tattered:327, trim:188
dress/  (52,411 files)  →  bodice:4,464, kimono:39,011, sash:4,472, slit:4,464
eyes/  (56,794 files)  →  cyclops:5,071, eyebrows:1,035, human:50,688
facial/  (351,280 files)  →  earrings:52,656, glasses:122,749, masks:18,720, monocle:31,150, patches:126,005
feet/  (301,302 files)  →  accessory:12,312, armour:2,960, boots:92,334, hoofs:7,912, sandals:23,034, shoes:92,900, slippers:17,450, socks:52,400
hair/  (67,894 files)  →  afro:352, balding:352, bangs:704, bangslong:352, bangslong2:516, bangsshort:352, bedhead:352, bob:704, braid:1,445, braid2:523, bunches:611, buzzcut:352, cornrows:352, cowlick:704, curls:2,249, curly:1,056, curtains:704, dreadlocks:704, extensions:7,580, flat:1,060, half:352, h

## Section 4 — 결과 확인

In [4]:
print(f"{'Category':<14} {'Subfolder':<20} {'Files':>8}")
print("-" * 46)

grand_total = 0

for cat_dir in sorted(PROCESSED_DIR.iterdir()):
    if not cat_dir.is_dir():
        continue
    if cat_dir.name in SKIP_CATEGORIES:
        body_count = sum(1 for f in cat_dir.iterdir() if f.is_file() and f.suffix == '.png')
        print(f"{'body':<14} {'(unchanged)':<20} {body_count:>8,}")
        grand_total += body_count
        continue

    subdirs = sorted([d for d in cat_dir.iterdir() if d.is_dir()])
    flat_files = sum(1 for f in cat_dir.iterdir() if f.is_file() and f.suffix == '.png')

    if not subdirs:
        # 분리 불필요했던 카테고리
        count = flat_files
        print(f"{cat_dir.name:<14} {'(flat)':<20} {count:>8,}")
        grand_total += count
    else:
        first = True
        for sub in subdirs:
            count = sum(1 for f in sub.iterdir() if f.is_file() and f.suffix == '.png')
            cat_label = cat_dir.name if first else ''
            print(f"{cat_label:<14} {sub.name:<20} {count:>8,}")
            grand_total += count
            first = False
        if flat_files > 0:
            print(f"{'':14} {'(미분류 잔여)':<20} {flat_files:>8,}")
            grand_total += flat_files

print("-" * 46)
print(f"{'TOTAL':<14} {'':<20} {grand_total:>8,}")

Category       Subfolder               Files
----------------------------------------------
arms           armour                    472
               bracers                   485
               hands                   1,014
               wrists                    985
backpack       backpack                5,880
               basket                  2,314
               jetpack                 1,686
               squarepack              5,920
               straps                  7,080
beards         beard                   1,644
               mustache                2,072
body           (unchanged)             4,544
cape           solid                     327
               tattered                  327
               trim                      188
dress          bodice                  4,464
               kimono                 39,011
               sash                    4,472
               slit                    4,464
eyes           cyclops                 5,071
        

## Section 5 — 산출물 저장

In [5]:
import io, sys

OUT_DIR = Path('output/03-2_keyword_split')
OUT_DIR.mkdir(parents=True, exist_ok=True)

buf = io.StringIO()
old_stdout = sys.stdout
sys.stdout = buf

print("=" * 50)
print("03-2 Keyword Split — Result")
print("=" * 50)
print()

for cat_dir in sorted(PROCESSED_DIR.iterdir()):
    if not cat_dir.is_dir():
        continue
    if cat_dir.name in SKIP_CATEGORIES:
        body_count = sum(1 for f in cat_dir.iterdir() if f.is_file() and f.suffix == '.png')
        print(f"body/ (unchanged): {body_count:,} files")
        continue

    subdirs = sorted([d for d in cat_dir.iterdir() if d.is_dir()])
    flat_files = sum(1 for f in cat_dir.iterdir() if f.is_file() and f.suffix == '.png')

    if not subdirs:
        count = flat_files
        print(f"{cat_dir.name}/ (flat): {count:,}")
    else:
        totals = []
        for sub in subdirs:
            count = sum(1 for f in sub.iterdir() if f.is_file() and f.suffix == '.png')
            totals.append(f"{sub.name}: {count:,}")
        print(f"{cat_dir.name}/ → " + ', '.join(totals))

sys.stdout = old_stdout

(OUT_DIR / 'text_output.txt').write_text(buf.getvalue(), encoding='utf-8')
print(buf.getvalue())
print(f"Saved: {OUT_DIR / 'text_output.txt'}")

03-2 Keyword Split — Result

arms/ → armour: 472, bracers: 485, hands: 1,014, wrists: 985
backpack/ → backpack: 5,880, basket: 2,314, jetpack: 1,686, squarepack: 5,920, straps: 7,080
beards/ → beard: 1,644, mustache: 2,072
body/ (unchanged): 4,544 files
cape/ → solid: 327, tattered: 327, trim: 188
dress/ → bodice: 4,464, kimono: 39,011, sash: 4,472, slit: 4,464
eyes/ → cyclops: 5,071, eyebrows: 1,035, human: 50,688
facial/ → earrings: 52,656, glasses: 122,749, masks: 18,720, monocle: 31,150, patches: 126,005
feet/ → accessory: 12,312, armour: 2,960, boots: 92,334, hoofs: 7,912, sandals: 23,034, shoes: 92,900, slippers: 17,450, socks: 52,400
hair/ → afro: 352, balding: 352, bangs: 704, bangslong: 352, bangslong2: 516, bangsshort: 352, bedhead: 352, bob: 704, braid: 1,445, braid2: 523, bunches: 611, buzzcut: 352, cornrows: 352, cowlick: 704, curls: 2,249, curly: 1,056, curtains: 704, dreadlocks: 704, extensions: 7,580, flat: 1,060, half: 352, halfmessy: 352, high: 874, idol: 352, jewfro: